In [1]:
import json
from math import radians, sin, cos, sqrt, atan2

# ---- Load Stage 1 data ----
with open("../outputs/incidents/stage1_frames.json") as f:
    stage1_frames = json.load(f)

# ---- Verification Agent ----
def gps_distance_meters(gps1, gps2):
    R = 6371000
    lat1, lon1 = radians(gps1[0]), radians(gps1[1])
    lat2, lon2 = radians(gps2[0]), radians(gps2[1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def build_incident(group):
    avg_conf = sum(d["conf"] for d in group) / len(group)
    return {
        "incident_id": f"RA-{group[0]['frame']:04d}",
        "anomaly": group[0]["class"],
        "status": "VERIFIED",
        "evidence_frames": len(group),
        "avg_confidence": round(avg_conf, 3),
        "gps": group[len(group)//2]["gps"],
        "frame_range": (group[0]["frame"], group[-1]["frame"])
    }

def verify_incidents(detections, min_conf=0.6, max_frame_gap=5, max_gps_dist=20, min_streak=3):
    detections = sorted(detections, key=lambda d: d["frame"])
    incidents = []
    current_group = []
    for det in detections:
        if det["conf"] < min_conf:
            continue
        if not current_group:
            current_group = [det]
            continue
        last = current_group[-1]
        same_class = det["class"] == last["class"]
        close_frame = (det["frame"] - last["frame"]) <= max_frame_gap
        close_gps = gps_distance_meters(det["gps"], last["gps"]) <= max_gps_dist
        if same_class and close_frame and close_gps:
            current_group.append(det)
        else:
            if len(current_group) >= min_streak:
                incidents.append(build_incident(current_group))
            current_group = [det]
    if len(current_group) >= min_streak:
        incidents.append(build_incident(current_group))
    return incidents

# ---- Context Agent ----
def context_agent(incident, weather, repetition_count=1):
    anomaly = incident["anomaly"]
    condition = weather.get("condition", "Unknown")
    is_rainy = condition in ("Rain", "Drizzle", "Thunderstorm")
    context = {
        "incident_id": incident["incident_id"], "anomaly": anomaly,
        "weather_condition": condition, "repetition_count": repetition_count,
        "conclusion": None, "risk_level": "Normal"
    }
    if "water" in anomaly.lower():
        if is_rainy:
            context["conclusion"] = "Water accumulation likely related to current rainfall"
        elif repetition_count > 1:
            context["conclusion"] = "Possible persistent drainage issue"
            context["risk_level"] = "Increased"
        else:
            context["conclusion"] = "Water presence unexplained by weather — needs inspection"
            context["risk_level"] = "Increased"
    elif "crack" in anomaly.lower():
        if is_rainy:
            context["conclusion"] = "Crack may worsen faster due to water seepage"
            context["risk_level"] = "Increased"
        elif repetition_count > 1:
            context["conclusion"] = "Recurring crack — monitor for expansion"
            context["risk_level"] = "Increased"
        else:
            context["conclusion"] = "Standard crack, no weather-related urgency"
    elif "pothole" in anomaly.lower():
        if is_rainy:
            context["conclusion"] = "Pothole with active rain — increased vehicle hazard"
            context["risk_level"] = "Increased"
        elif repetition_count > 1:
            context["conclusion"] = "Recurring pothole — likely worsening, needs prioritized repair"
            context["risk_level"] = "Increased"
        else:
            context["conclusion"] = "Pothole detected, standard road hazard"
    else:
        context["conclusion"] = "General road anomaly detected"
    return context

# ---- Severity Agent ----
def severity_agent(incident, context_result, repetition_count=1):
    score = 0
    score += incident["avg_confidence"] * 30
    score += min(incident["evidence_frames"], 10) * 2
    risk_weights = {"Normal": 0, "Increased": 25}
    score += risk_weights.get(context_result["risk_level"], 0)
    score += min(repetition_count, 5) * 8

    if score >= 70:
        severity = "CRITICAL"
    elif score >= 50:
        severity = "HIGH"
    elif score >= 30:
        severity = "MEDIUM"
    else:
        severity = "LOW"

    return {"incident_id": incident["incident_id"], "severity_score": round(score, 1), "severity": severity}

print("All functions loaded.")

All functions loaded.


In [2]:
sample_detections = [
    {"frame": 118, "class": "D40_pothole", "conf": 0.71, "gps": (13.04500, 80.21100), "bbox": [120,200,180,260]},
    {"frame": 119, "class": "D40_pothole", "conf": 0.88, "gps": (13.04502, 80.21102), "bbox": [122,198,182,258]},
    {"frame": 120, "class": "D40_pothole", "conf": 0.94, "gps": (13.04504, 80.21104), "bbox": [124,199,184,259]},
    {"frame": 121, "class": "D40_pothole", "conf": 0.92, "gps": (13.04506, 80.21106), "bbox": [125,200,185,260]},
]

incidents = verify_incidents(sample_detections)
test_incident = incidents[0]
test_weather = stage1_frames[0]["weather"]

context_result = context_agent(test_incident, test_weather, repetition_count=3)
severity_result = severity_agent(test_incident, context_result, repetition_count=3)

print("Context:", context_result)
print("Severity:", severity_result)

Context: {'incident_id': 'RA-0118', 'anomaly': 'D40_pothole', 'weather_condition': 'Clouds', 'repetition_count': 3, 'conclusion': 'Recurring pothole — likely worsening, needs prioritized repair', 'risk_level': 'Increased'}
Severity: {'incident_id': 'RA-0118', 'severity_score': 82.9, 'severity': 'CRITICAL'}


In [3]:
low_severity_incident = {
    "incident_id": "RA-0200",
    "anomaly": "D00_longitudinal_crack",
    "avg_confidence": 0.62,
    "evidence_frames": 3,
    "gps": (13.06, 80.22)
}

low_context = context_agent(low_severity_incident, {"condition": "Clear"}, repetition_count=1)
low_severity = severity_agent(low_severity_incident, low_context, repetition_count=1)

print("Context:", low_context)
print("Severity:", low_severity)

Context: {'incident_id': 'RA-0200', 'anomaly': 'D00_longitudinal_crack', 'weather_condition': 'Clear', 'repetition_count': 1, 'conclusion': 'Standard crack, no weather-related urgency', 'risk_level': 'Normal'}
Severity: {'incident_id': 'RA-0200', 'severity_score': 32.6, 'severity': 'MEDIUM'}


In [4]:
DEPARTMENT_MAP = {
    "D00_longitudinal_crack": "Road Maintenance",
    "D10_transverse_crack": "Road Maintenance",
    "D20_alligator_crack": "Road Maintenance",
    "D40_pothole": "Road Maintenance",
    "repair": None,
}

def routing_agent(incident, severity_result):
    anomaly = incident["anomaly"]
    department = DEPARTMENT_MAP.get(anomaly, "General Public Works")

    if department is None:
        return {
            "incident_id": incident["incident_id"],
            "status": "IGNORED",
            "reason": "Marked as already-repaired section, no action needed"
        }

    if severity_result["severity"] == "CRITICAL":
        action = "Immediate inspection and repair required within 24 hours"
    elif severity_result["severity"] == "HIGH":
        action = "Priority inspection within 3-5 days"
    elif severity_result["severity"] == "MEDIUM":
        action = "Schedule routine inspection within 2 weeks"
    else:
        action = "Log for periodic maintenance review"

    return {
        "incident_id": incident["incident_id"],
        "department": department,
        "severity": severity_result["severity"],
        "recommended_action": action
    }

routing_critical = routing_agent(test_incident, severity_result)
routing_medium = routing_agent(low_severity_incident, low_severity)

print("Critical case routing:", routing_critical)
print("Medium case routing:", routing_medium)

Critical case routing: {'incident_id': 'RA-0118', 'department': 'Road Maintenance', 'severity': 'CRITICAL', 'recommended_action': 'Immediate inspection and repair required within 24 hours'}
Medium case routing: {'incident_id': 'RA-0200', 'department': 'Road Maintenance', 'severity': 'MEDIUM', 'recommended_action': 'Schedule routine inspection within 2 weeks'}
